# Notebook 04: Full PCA Pipeline Walkthrough
**AAIRM** — Agentic AI Inventory Replenishment and Management  
Paper: Syed et al. (2025), *Agentic Commerce*, Frontiers in [Journal].

Traces all 18 interaction steps from Figure 2 of the paper (sequence diagram) for SKU F123 (frozen category).

In [ ]:
import syssys.path.insert(0, '..')from aairm.utils.seed import set_global_seedfrom aairm.utils.config import AAIRMConfigfrom aairm.simulation.environment import RetailEnvfrom aairm.agents.meta_orchestrator import MetaOrchestratorfrom aairm.agents.base import AgentStatefrom aairm.models.forecasting.naive_forecaster import NaiveForecasterset_global_seed(42)print('Imports OK')

## Setup — Build a small environment for demonstration

In [ ]:
config = AAIRMConfig()config.simulation.n_skus = 50config.simulation.test_horizon_days = 7env = RetailEnv(config.simulation)env.reset()print(f'Environment ready: {config.simulation.n_skus} SKUs')

## Steps 1–2: User Request + Orchestrator Dispatch

In [ ]:
print('Step 1: User prompt: Ensure frozen SKU is always in stock')print('Step 2: Meta-Orchestrator dispatches to P1 Inventory Monitor')

## Steps 3–5: P1 Inventory Monitor

In [ ]:
orch = MetaOrchestrator(config=config, erp_backend=env, supplier_backend=env, trend_backend=env, forecaster=NaiveForecaster())state = AgentState(day=0)state = orch.p1.run(state)print(f'Step 3: inventory.read() called')print(f'Step 4: on_hand snapshot retrieved for {len(state.sku_inventory_snapshot)} SKUs')print(f'Step 5: Low-stock SKUs: {len(state.low_stock_skus)}')if state.low_stock_skus: print(f'  First low-stock: {state.low_stock_skus[0]}')

## Steps 6–8: C1 Context + Forecasting, C2 Reorder Optimisation

In [ ]:
state = orch.p4.run(state)state = orch.c1.run(state)print(f'Step 6: Demand history retrieved; Step 7: 7-day forecast computed')if state.demand_forecasts:    ex_sku = list(state.demand_forecasts.keys())[0]    fc = state.demand_forecasts[ex_sku]    print(f'  {ex_sku}: mean={fc["mean"]:.1f} p10={fc["p10"]:.1f} p90={fc["p90"]:.1f}')state = orch.c2.run(state)print(f'Step 8: Optimal Q* computed for {len(state.order_proposals)} SKUs')

## Steps 9–13: C3 Supplier Ranking + C4 Negotiation + C5 Governance

In [ ]:
state = orch.c3.run(state)print(f'Step 9-11: Supplier catalogue queried; {len(state.supplier_rankings)} SKUs ranked')state = orch.c4.run(state)print(f'Step 12-13: Negotiation complete; {len(state.negotiated_terms)} terms finalised')state = orch.c5.run(state)print(f'Governance: {len(state.approved_orders)} orders approved')

## Steps 14–18: A1 Order Execution + ERP Update + User Response

In [ ]:
state = orch.a1.run(state)print(f'Step 14: PO created and submitted to supplier')print(f'Step 15-16: PO confirmed; ERP updated')print(f'Step 17: POs issued: {state.purchase_orders_issued}')print(f'Step 18: Meta-Orchestrator response: {state.summary()}')